# Access to Alerts at USDF and view images and Kernels

- **Author:** Sylvie Dagoret-Campagne  
- **Creation date:** 2026-04-09
- **Last update:** 2026-04-12
- **Environment:** USDF RSP — `LSST` kernel (`lsst_distrib` stack)


## 1. Imports

Standard scientific stack plus the LSST middleware:  
- `lsst.daf.butler` — data access layer  
- `lsst.afw.display` — image display utilities  
- `lsst.geom` — sky coordinate geometry  
- `lsst.skymap` — tract/patch sky tessellation


In [ ]:
import lsst.pipe.base

print(lsst.pipe.base.__version__)

In [ ]:
import sys
import matplotlib.pyplot as plt
import lsst.afw.display as afwDisplay
from lsst.geom import SpherePoint, degrees
from lsst.afw.image import ExposureF
from lsst.skymap import PatchInfo, Index2D
import numpy as np
import pandas as pd
from astropy.time import Time

# %matplotlib widget
import seaborn as sns
# from lsst.analysis.ap import apdb

import lsst.afw.display as afwDisplay
from firefly_client import FireflyClient

import lsst.afw.image as afwImage
import math

In [ ]:
# LSST data access middleware
from lsst.daf.butler import Butler

In [ ]:
def get_uris_from_butler(butler, datasetType, collections=None, **query_kwargs):
    """
    Return a list of URIs for a given datasetType in a Butler collection.

    Parameters
    ----------
    butler : lsst.daf.butler.Butler
        The Butler object
    datasetType : str
        The dataset type, e.g. 'raw', 'calexp'
    collections : list of str, optional
        Butler collections to query
    query_kwargs : dict
        Additional query parameters, e.g. visit=123, filter='r'

    Returns
    -------
    uris : list of str
        File paths of the datasets
    """
    refs = butler.registry.queryDatasets(datasetType, collections=collections, **query_kwargs)
    return [butler.getURI(ref) for ref in refs]

In [ ]:
def get_refs_from_butler(butler, datasetType, collections=None, return_uris=False, **query_kwargs):
    """
    Get dataset references (or URIs) from a Butler, safely handling numpy types.

    Parameters
    ----------
    butler : lsst.daf.butler.Butler
        Butler instance.
    datasetType : str
        Dataset type to query (e.g., "calexp", "raw", "src").
    collections : str or list of str, optional
        Butler collections to query.
    return_uris : bool, default False
        If True, return URIs instead of refs.
    **query_kwargs :
        Query parameters, e.g., visit=[...], ccd=[...], filter=['r','g'].

    Returns
    -------
    list of DatasetRef or list of str
        List of references or URIs.
    """

    # Convert numpy types to Python native types to avoid Butler query errors
    safe_kwargs = {}
    for k, v in query_kwargs.items():
        if isinstance(v, np.ndarray):
            safe_kwargs[k] = [int(x) if np.issubdtype(type(x), np.integer) else x for x in v]
        elif isinstance(v, (np.integer, np.int64, np.int32)):
            safe_kwargs[k] = int(v)
        else:
            safe_kwargs[k] = v

    # Query dataset references
    refs = butler.registry.queryDatasets(datasetType, collections=collections, **safe_kwargs)

    if return_uris:
        return [butler.getURI(ref) for ref in refs]

    return refs

## 2. Configuration

Select the Butler repository, the DRP processing collection, the sky map name,  
and the instrument.  Multiple DRP collections are listed for reference:  
uncomment the desired one before running.


- See collection here : https://usdf-rsp.slac.stanford.edu/plot-navigator
- See Campaign : https://rubinobs.atlassian.net/wiki/spaces/DM/pages/661192727/LSSTCam+Intermittent+DRP+Runs
- See the bulter lists : https://developer.lsst.io/usdf/storage.html
- See Prompt+Processing+with+LSSTCam : https://rubinobs.atlassian.net/wiki/spaces/DM/pages/1417609236/Prompt+Processing+with+LSSTCam+Daytime+AP+2026#
- Examples of Notebooks : https://github.com/lsst-so/notebooks_dia/tree/main

In [ ]:
# ── Butler repository and DRP collection ──────────────────────────────────────
REPO = "main"
collection = "LSSTCam/prompt/output-2026-02-26"
skymapName = "lsst_cells_v2"
instrument = "LSSTCam"

# ── Date range for exposure queries ───────────────────────────────────────────
date_start = 20260101  # YYYYMMDD — earliest day_obs to include
date_selection = 20260409  # YYYYMMDD — reference date for the analysis

# ── Build WHERE clauses for Butler registry queries ───────────────────────────
where_clause = "instrument = '" + f"{instrument}" + "'"
where_clause_date = where_clause + f"and day_obs >= {date_start}"

# ── Instantiate the Butler and its registry ───────────────────────────────────
butler = Butler(REPO, collections=collection)
registry = butler.registry

In [ ]:
# Retrieve the sky tessellation (skyMap) from the butler.
# The 'lsst_cells_v2' skymap divides the full sky into tracts and patches.
# Error handling is included because the skymap may not be present in all collections.
try:
    skymap = butler.get("skyMap", skymap=skymapName, collections=collection)
except Exception as inst:
    print(type(inst))  # exception type
    print(inst.args)  # arguments stored in .args
    print(inst)  # full message via __str__

In [ ]:
# Inspect the skymap object (number of tracts, pixel scale, etc.)
# skymap

In [ ]:
FLAGS_DEBUG_SCHEMA = True

## 8. Available dataset types in the collection

List all dataset types present in the selected collection,  
filtering out pipeline-internal products (configs, logs, metrics, plots)  
to focus on science data products (calexp, src, dia_source, dia_object, …).


In [ ]:
FLAG_DUMP_DATASETS = True
if FLAG_DUMP_DATASETS:
    for datasetType in registry.queryDatasetTypes():
        if registry.queryDatasets(datasetType, collections=collection).any(execute=False, exact=False):
            # Skip pipeline bookkeeping products
            if (
                ("_config" not in datasetType.name)
                and ("_log" not in datasetType.name)
                and ("_metadata" not in datasetType.name)
                and ("_resource_usage" not in datasetType.name)
                and ("Plot" not in datasetType.name)
                and ("Metric" not in datasetType.name)
                and ("metric" not in datasetType.name)
            ):
                print(datasetType)

In [ ]:
# Inspect the required dimensions for dia_source (needed for difference imaging queries)
print(butler.registry.getDatasetType("dia_forced_source_apdb").dimensions)

In [ ]:
# Inspect the required dimensions for dia_object
print(butler.registry.getDatasetType("dia_object_apdb").dimensions)

In [ ]:
# Inspect the required dimensions for dia_source (needed for difference imaging queries)
print(butler.registry.getDatasetType("dia_source_apdb").dimensions)

## Example of Light curve subsample in Fink-Broker data

```
 	diaObj 	diaSrc 	mjd 	band 	visit 	detector 	x 	y 	psfFlux 	psfFluxErr 	scienceFlux 	scienceFluxErr 	templateFlux 	templateFluxErr 	day_obs 	month_obs
1029 	313853517444939794 	170063690301177938 	61098.108349 	y 	2026022600207 	51 	145.16164 	3639.300500 	10921.9270 	1243.32760 	44045.580 	1226.78440 	32617.346 	274.21304 	20260226 	202602
1030 	313853517444939794 	170063690452172875 	61098.108813 	y 	2026022600208 	83 	3827.32930 	2687.968000 	9844.3880 	1189.71690 	42716.867 	1176.07300 	32472.490 	273.70413 	20260226 	202602
1031 	313853517444939794 	170063690858496056 	61098.112664 	y 	2026022600211 	90 	2788.16190 	1668.272300 	10380.5080 	1019.47400 	43284.140 	1013.96420 	31991.410 	376.74704 	20260226 	202602
1032 	313853517444939794 	170063690994286595 	61098.113128 	y 	2026022600212 	93 	1259.44450 	41.697483 	10514.9760 	1008.98395 	42434.120 	998.61346 	30744.092 	393.22940 	20260226 	202602
1033 	313853517444939794 	170063723341807659 	61098.270223 	y 	2026022600453 	95 	3550.78900 	1363.893300 	8229.4970 	1017.97644 	40883.297 	1023.03370 	31669.326 	411.03214 	20260226 	202602
1034 	313853517444939794 	170063723474452578 	61098.270691 	y 	2026022600454 	92 	2100.58940 	2938.168500 	9661.8140 	1010.44160 	41524.082 	1016.33514 	30815.621 	452.85654 	20260226 	202602
1035 	313853517444939794 	170063723613913266 	61098.272697 	r 	2026022600455 	102 	2612.80860 	3824.363000 	8016.3496 	189.21060 	25180.540 	187.87000 	17067.248 	39.82949 	20260226 	202602
1036 	313853517444939794 	170063723748130852 	61098.273166 	r 	2026022600456 	102 	1098.03020 	841.125730 	7928.1895 	188.90501 	25311.723 	189.69778 	17217.742 	64.94376 	20260226 	202602
1037 	313853517444939794 	170063723878678545 	61098.275115 	z 	2026022600457 	95 	1300.34720 	544.580300 	9032.4520 	417.41107 	30190.838 	413.37164 	21071.193 	122.68716 	20260226 	202602
1038 	313853517444939794 	170063724011323506 	61098.275586 	z 	2026022600458 	92 	3572.64770 	3112.698700 	9296.9560 	409.83118 	30139.334 	404.60516 	20683.617 	138.13329 	20260226 	202602

```

### Visit in Fink-Broker

In [ ]:
diaObjectId = 313853517444939794
dataId = {"instrument": instrument, "band": "r", "detector": 102, "visit": 2026022600455}

In [ ]:
t = butler.get("dia_source_apdb", dataId)

In [ ]:
t[t["diaObjectId"] == diaObjectId]

## View the Science, DIA and Templates images

In [ ]:
try:
    the_singlevisitimage = butler.get("preliminary_visit_image", dataId)
    the_differenceimage = butler.get("difference_image", dataId)
    the_template_detector = butler.get("template_detector", dataId)
    title = f"{dataId['visit']},{dataId['detector']},{dataId['band']}"
    x, y = t["x"][0], t["y"][0]
    list_of_points = [(x, y)]
except Exception as e:
    # compact but useful log
    print(f"[FAIL] → visit={dataId['visit']} det={dataId['detector']} band={dataId['band']}")
    print(f"       {type(e).__name__}: {e}")

In [ ]:
all_singlevisitimages = [the_singlevisitimage]
all_differenceimages = [the_differenceimage]
all_template_detector = [the_template_detector]

all_titles = [title]
all_datapoints = [list_of_points]
N = len(all_singlevisitimages)
print(f"Number of image to plot : {N}")
count = 0

### Plot in firefly

In [ ]:
afwDisplay.setDefaultBackend("firefly")

In [ ]:
for count in range(N):
    # display the science image
    display = afwDisplay.Display(frame=count + 1)
    # cannot succeed to show white stars on dark sky
    # display.setImageColormap('gray')
    display.scale("asinh", "zscale")
    display.mtv(all_singlevisitimages[count].image, title=all_titles[count])
    list_of_points = all_datapoints[count]
    with display.Buffering():
        #
        for x, y in list_of_points:
            display.dot("o", x, y, size=30, ctype="red")

    # display the difference image
    display = afwDisplay.Display(frame=count + 2)
    display.scale("asinh", "zscale")
    display.mtv(all_differenceimages[count].image, title=all_titles[count] + "_dia")
    list_of_points = all_datapoints[count]
    with display.Buffering():
        #
        for x, y in list_of_points:
            display.dot("o", x, y, size=30, ctype="blue")

    # display the template image
    display = afwDisplay.Display(frame=count + 3)
    display.scale("asinh", "zscale")
    display.mtv(all_template_detector[count].image, title=all_titles[count] + "_temp")
    list_of_points = all_datapoints[count]
    with display.Buffering():
        #
        for x, y in list_of_points:
            display.dot("o", x, y, size=30, ctype="green")

In [ ]:
# display.clearViewer()

## Fetch more information

### Get `single_visit_star_footprints`

In [ ]:
try:
    the_singlevisit_star_footprints = butler.get("single_visit_star_footprints", dataId)
except Exception as e:
    # compact but useful log
    print(f"[FAIL] → visit={dataId['visit']} det={dataId['detector']} band={dataId['band']}")
    print(f"       {type(e).__name__}: {e}")

In [ ]:
if FLAGS_DEBUG_SCHEMA:
    schema = the_singlevisit_star_footprints.schema

    for item in schema:
        field = item.field
        name = field.getName()
        dtype = field.getTypeString()
        doc = field.getDoc()

        print(f"{name:30s} | type={dtype:10s} | doc={doc}")

### Get `difference_kernel`


See some reference here : https://dmtn-021.lsst.io/v/v1/index.html

$$ K(x,y;u,v)=\sum_i​a_i(u,v)K_i\times(x,y)$$

🧱 1. Internal structure of the difference_kernel

A LinearCombinationKernel in difference imaging is:

$$ K(x,y;u,v)=\sum_i​a_i(u,v)K_i(x,y)$$

Objects like `lsst.afw.math.LinearCombinationKernel` are somewhat opaque when printed directly, but their full internal structure can be extracted.

---

#### 🔎 1. Simple inspection

```python
print(kernel)
```

👉 Generally uninformative (just type + memory address).

---

#### 🧱 2. Accessing kernel components

A `LinearCombinationKernel` is a combination:

$$
K(x, y) = \sum_i a_i \, K_i(x, y)
$$

You can retrieve:

```python
basis_kernels = kernel.getBasisList()
coeffs = kernel.getKernelParameters()
```

---

#### 📋 3. Detailed display

```python
for i, (k, a) in enumerate(zip(basis_kernels, coeffs)):
    print(f"--- Kernel {i} ---")
    print(f"Coefficient: {a}")
    print(f"Type: {type(k)}")
    print(f"Dimensions: {k.getWidth()} x {k.getHeight()}")
```

---

#### 🔬 4. Visualise each kernel (very useful)

```python
import numpy as np
import matplotlib.pyplot as plt

for i, k in enumerate(basis_kernels):
    arr = k.computeImage().getArray()
    plt.imshow(arr, origin="lower")
    plt.colorbar()
    plt.title(f"Kernel {i}")
    plt.show()
```

👉 This shows the basis functions (often Gaussians, shapelets, etc.)

---

#### ⚙️ 5. View the combined kernel

```python
image = kernel.computeImage()
arr = image.getArray()

plt.imshow(arr, origin="lower")
plt.colorbar()
plt.title("LinearCombinationKernel")
plt.show()
```

---

#### 🧠 Interpretation (important)

In Rubin/LSST, this type of kernel is commonly used for:

- modelling the **spatial PSF**
- convolution (PSF matching)
- interpolation across the focal plane

Typically:

- `basis_kernels` = basis set (Gaussians, PCA, shapelets…)
- `coeffs` = spatial dependence or local fit

---

#### ⚠️ Subtle detail

Depending on context (PSF, warping, coadd):

- coefficients can be **fixed**
- or position-dependent (via `SpatialFunction` elsewhere)

---

👉 This gives the **local PSF matching kernel**

---

#### 🧠 7. Physical interpretation (important for your work)

In the Rubin difference imaging context, this kernel transforms:

```
template PSF → science PSF
```

It therefore encodes:
- seeing variation
- anisotropies
- optical effects (indirect vignetting via PSF shape)

---

##### ⚠️ 8. Key methods

##### 🔹 Structure

- `getKernelList()` → basis kernels
- `getNBasisKernels()` → number of basis kernels

##### 🔹 Coefficients

- `getKernelParameters()` → current coefficients
- `getSpatialFunctionList()` → spatial variation functions

##### 🔹 Evaluation

- `computeImage(...)` → compute the concrete kernel image

##### 🔹 Spatial check

- `isSpatiallyVarying()` → very important flag

---

#### 🚀 9. Advanced diagnostics

##### Check whether the kernel is dominated by a few modes

```python
import numpy as np
coeffs = np.array(kernel.getKernelParameters())
print(coeffs / coeffs.sum())
```

##### Inspect each basis individually

```python
for i, k in enumerate(kernel.getKernelList()):
    img = k.computeImage().getArray()
    plt.imshow(img, origin="lower")
    plt.title(f"Basis {i}")
    plt.colorbar()
    plt.show()
```

---

#### 🧭 10. What you can do with it (highly relevant)

This kernel allows you to:

- analyse **spatial PSF variation**
- understand subtraction residuals
- diagnose issues such as PSF mismatch, optical gradients, or calibration defects

---


In [ ]:
try:
    the_difference_kernel = butler.get("difference_kernel", dataId)
except Exception as e:
    # compact but useful log
    print(f"[FAIL] → visit={dataId['visit']} det={dataId['detector']} band={dataId['band']}")
    print(f"       {type(e).__name__}: {e}")

In [ ]:
kernel = the_difference_kernel

print("Nb basis kernels:", kernel.getNBasisKernels())
print("Nb kernel params:", kernel.getNKernelParameters())
print("Spatially varying:", kernel.isSpatiallyVarying())

In [ ]:
basis = kernel.getKernelList()

for i, k in enumerate(basis):
    print(f"Kernel {i}: {type(k)} | size = {k.getWidth()}x{k.getHeight()}")

In [ ]:
# kernel.getSpatialFunctionList()

### Select the kernel at the position of the dia object

In [ ]:
# x, y = 1000, 2000  # position pixel
x, y = all_datapoints[0][0][0], all_datapoints[0][0][1]

img = afwImage.ImageD(kernel.getDimensions())
kernel.computeImage(img, doNormalize=True, x=x, y=y)

arr = img.getArray()

In [ ]:
plt.imshow(arr, origin="lower")
plt.colorbar()
plt.title(f"Kernel at ({x:.3f},{y:.3f})")
plt.show()

In [ ]:
coeffs = np.array(kernel.getKernelParameters())
print(coeffs / coeffs.sum())

### Plot all the basis function Not normalised

In [ ]:
kernels = kernel.getKernelList()
n = len(kernels)

# Define a "square" grid layout
ncols = math.ceil(np.sqrt(n))
nrows = math.ceil(n / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))
axes = np.atleast_1d(axes).ravel()

for i, (ax, k) in enumerate(zip(axes, kernels)):
    img = afwImage.ImageD(k.getDimensions())
    k.computeImage(img, doNormalize=True)
    arr = img.getArray()

    im = ax.imshow(arr, origin="lower")
    ax.set_title(f"Basis {i}")
    ax.axis("off")

    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

# Disable unused axes
for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.suptitle("Basis of unknormalized kernel functions", fontsize=20, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
kernels = kernel.getKernelList()
n = len(kernels)

# Define a "square" grid layout
ncols = math.ceil(np.sqrt(n))
nrows = math.ceil(n / ncols)

# Compute global min/max across all basis kernels
all_arrays = []
for k in kernels:
    img = afwImage.ImageD(k.getDimensions())
    k.computeImage(img, True)
    all_arrays.append(img.getArray())

vmin = min(arr.min() for arr in all_arrays)
vmax = max(arr.max() for arr in all_arrays)

fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows), layout="constrained")
axes = np.atleast_1d(axes).ravel()

for i, (ax, arr) in enumerate(zip(axes, all_arrays)):
    im = ax.imshow(arr, origin="lower", vmin=vmin, vmax=vmax)
    ax.set_title(f"Basis {i}")
    ax.axis("off")

# shared colorbar
fig.colorbar(im, ax=axes.tolist(), shrink=0.6)

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.suptitle("Basis of normalized kernel functions", fontsize=20, fontweight="bold")
# plt.tight_layout()
plt.show()